In [1]:
import pandas as pd
import plotly.express as px
import json
import plotly.io as pio

# Evita el error de conexión abriendo directamente en el navegador
pio.renderers.default = "browser"

In [2]:
# 1. CONFIGURACIÓN DE RUTAS (Ajusta si los nombres de archivos cambian)
path_csv = r'C:\Users\Cami\Downloads\Mapas2\Accidentes.csv'
path_json_coords = r'C:\Users\Cami\Downloads\Mapas2\cr.json'

In [3]:
# 2. CARGA Y PREPARACIÓN DE DATOS
# Cargamos el CSV y el JSON de coordenadas
df = pd.read_csv(path_csv, sep=';')
with open(path_json_coords, encoding='utf-8') as f:
    coords_data = json.load(f)

In [9]:
# Limpiamos TODAS las columnas (quitamos espacios al inicio y final)
df.columns = df.columns.str.strip()

# Limpiamos el contenido de las columnas de ubicación para que coincidan con el JSON
df['Provincia'] = df['Provincia'].str.strip()
df['Cantón'] = df['Cantón'].str.strip()

In [11]:
# Convertimos el JSON de coordenadas en un DataFrame para cruzarlo# Preparamos el DataFrame de coordenadas
df_coords = pd.DataFrame(coords_data)
df_coords['city'] = df_coords['city'].str.strip()


In [5]:
# Limpieza: quitamos espacios y normalizamos a mayúsculas para que el cruce sea perfecto
df.columns = df.columns.str.strip()
df['Cantón_Clean'] = df['Cantón'].str.strip().str.upper()
df_coords['city_clean'] = df_coords['city'].str.upper()

In [13]:
# 3. CRUCE DE DATOS (Merge)
# Unimos por Cantón (CSV) y City (JSON)
df_final = pd.merge(
    df,
    df_coords[['city', 'lat', 'lng']],
    left_on='Cantón',
    right_on='city',
    how='left'
)

In [14]:
# Convertimos coordenadas a números y quitamos filas sin ubicación para los mapas
df_final['lat'] = pd.to_numeric(df_final['lat'])
df_final['lng'] = pd.to_numeric(df_final['lng'])
df_geo = df_final.dropna(subset=['lat', 'lng'])

print("Datos listos y limpios. Generando mapas...")

Datos listos y limpios. Generando mapas...


In [15]:
df_tree = df_geo.groupby(['Provincia', 'Cantón', 'Clase de accidente']).size().reset_index(name='Total')

fig_tree = px.treemap(
    df_tree,
    path=['Provincia', 'Cantón', 'Clase de accidente'],
    values='Total',
    color='Clase de accidente',
    color_discrete_map={'Solo heridos leves': '#2980b9', 'Con muertos o graves': '#c0392b'},
    title='Jerarquía de Accidentes: Provincia -> Cantón'
)
fig_tree.show()

In [16]:
df_scatter = df_geo.groupby(['Año', 'Provincia', 'Cantón', 'Clase de accidente', 'lat', 'lng']).size().reset_index(
    name='Total')

fig_scatter = px.scatter_mapbox(
    df_scatter,
    lat="lat", lon="lng",
    size="Total",
    color="Clase de accidente",
    animation_frame="Año",
    hover_name="Cantón",
    zoom=7,
    size_max=30,
    mapbox_style="carto-positron",
    color_discrete_map={'Solo heridos leves': '#2980b9', 'Con muertos o graves': '#c0392b'},
    title='Evolución de Accidentes por Cantón (2018-2024)'
)
fig_scatter.show()

C:\Users\Cami\AppData\Local\Temp\ipykernel_23164\2647339431.py:4: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig_scatter = px.scatter_mapbox(


In [18]:
# --- MEJORA PARA EL MAPA DE CALOR ---
# Reducimos los datos agrupándolos para que el navegador no se sature
df_heat = df_geo.groupby(['Año', 'lat', 'lng']).size().reset_index(name='Intensidad')

fig_heat = px.density_mapbox(
    df_heat,
    lat='lat',
    lon='lng',
    z='Intensidad',
    radius=15,
    center=dict(lat=9.7489, lon=-83.7534),
    zoom=7,
    mapbox_style="carto-positron", # Estilo más ligero que stamen-terrain
    animation_frame="Año",
    title='Mapa de Calor: Concentración de Accidentes en Costa Rica'
)

# SOLUCIÓN AL ERROR DE CONEXIÓN: Guardar y abrir manualmente
nombre_archivo = "mapa_calor_accidentes.html"
fig_heat.write_html(nombre_archivo)

print(f"¡Mapa generado con éxito! Busca el archivo '{nombre_archivo}' en tu carpeta de proyecto y ábrelo con Chrome o Edge.")

¡Mapa generado con éxito! Busca el archivo 'mapa_calor_accidentes.html' en tu carpeta de proyecto y ábrelo con Chrome o Edge.


C:\Users\Cami\AppData\Local\Temp\ipykernel_23164\1263797699.py:5: DeprecationWarning: *density_mapbox* is deprecated! Use *density_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig_heat = px.density_mapbox(
